In [1]:
# Téléchargement des datasets
import urllib.request
import os

os.makedirs('../datasets/raw', exist_ok=True)

# Dataset 1 — Heart Disease (UCI)
url1 = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
urllib.request.urlretrieve(url1, '../datasets/raw/heart_disease.csv')
print("✓ Heart Disease téléchargé")

# Dataset 2 — Pima Indians Diabetes
url2 = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
urllib.request.urlretrieve(url2, '../datasets/raw/diabetes.csv')
print("✓ Diabetes téléchargé")

print("\nTous les datasets sont prêts !")

✓ Heart Disease téléchargé
✓ Diabetes téléchargé

Tous les datasets sont prêts !


In [2]:
import pandas as pd
import numpy as np

# Charger Heart Disease
col_names = ['age','sex','cp','trestbps','chol','fbs','restecg',
             'thalach','exang','oldpeak','slope','ca','thal','target']
df_heart = pd.read_csv('../datasets/raw/heart_disease.csv', 
                        names=col_names, na_values='?')

# Charger Diabetes
col_diabetes = ['pregnancies','glucose','blood_pressure','skin_thickness',
                'insulin','bmi','diabetes_pedigree','age','target']
df_diabetes = pd.read_csv('../datasets/raw/diabetes.csv', names=col_diabetes)

print("=== HEART DISEASE ===")
print(f"Lignes : {df_heart.shape[0]} | Colonnes : {df_heart.shape[1]}")
print(f"Valeurs manquantes : {df_heart.isnull().sum().sum()}")
print(f"Distribution cible :\n{df_heart['target'].value_counts()}")

print("\n=== DIABETES ===")
print(f"Lignes : {df_diabetes.shape[0]} | Colonnes : {df_diabetes.shape[1]}")
print(f"Valeurs manquantes : {df_diabetes.isnull().sum().sum()}")
print(f"Distribution cible :\n{df_diabetes['target'].value_counts()}")

=== HEART DISEASE ===
Lignes : 303 | Colonnes : 14
Valeurs manquantes : 6
Distribution cible :
target
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

=== DIABETES ===
Lignes : 768 | Colonnes : 9
Valeurs manquantes : 0
Distribution cible :
target
0    500
1    268
Name: count, dtype: int64


In [3]:
# Nettoyage Heart Disease
# Convertir la cible en binaire (0 = sain, 1 = malade)
df_heart['target'] = (df_heart['target'] > 0).astype(int)

# Sauvegarder les versions nettoyées
df_heart.to_csv('../datasets/processed/heart_disease_clean.csv', index=False)
df_diabetes.to_csv('../datasets/processed/diabetes_clean.csv', index=False)

print("=== HEART DISEASE (après nettoyage) ===")
print(f"Distribution cible : {df_heart['target'].value_counts().to_dict()}")
print(f"Valeurs manquantes par colonne :")
print(df_heart.isnull().sum()[df_heart.isnull().sum() > 0])

print("\n=== DIABETES (après nettoyage) ===")
print(f"Distribution cible : {df_diabetes['target'].value_counts().to_dict()}")

print("\nDatasets sauvegardés dans datasets/processed/")

=== HEART DISEASE (après nettoyage) ===
Distribution cible : {0: 164, 1: 139}
Valeurs manquantes par colonne :
ca      4
thal    2
dtype: int64

=== DIABETES (après nettoyage) ===
Distribution cible : {0: 500, 1: 268}

Datasets sauvegardés dans datasets/processed/


In [4]:
from ydata_profiling import ProfileReport
import os

os.makedirs('../reports', exist_ok=True)

# Rapport Heart Disease
print("Génération du rapport Heart Disease... (1-2 minutes)")
profile_heart = ProfileReport(
    df_heart,
    title="EDA — Heart Disease Dataset",
    explorative=True
)
profile_heart.to_file('../reports/eda_heart_disease.html')
print("✓ Rapport Heart Disease sauvegardé dans reports/")

# Rapport Diabetes
print("Génération du rapport Diabetes... (1-2 minutes)")
profile_diabetes = ProfileReport(
    df_diabetes,
    title="EDA — Diabetes Dataset",
    explorative=True
)
profile_diabetes.to_file('../reports/eda_diabetes.html')
print("✓ Rapport Diabetes sauvegardé dans reports/")

print("\nTous les rapports sont générés !")

C:\Users\leche\automl-platform\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\leche\AppData\Local\Temp\ipykernel_13964\2063812844.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Génération du rapport Heart Disease... (1-2 minutes)


Export report to file: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 38.25it/s]


✓ Rapport Heart Disease sauvegardé dans reports/
Génération du rapport Diabetes... (1-2 minutes)


Export report to file: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 14.22it/s]

✓ Rapport Diabetes sauvegardé dans reports/

Tous les rapports sont générés !


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("=" * 50)
print("MODÈLE BASELINE — Random Forest par défaut")
print("=" * 50)

# Préparer les données Heart Disease
X = df_heart.drop('target', axis=1)
y = df_heart['target']

# Gérer les valeurs manquantes
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y)

# Entraîner le modèle baseline
import time
start = time.time()
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train, y_train)
baseline_time = round(time.time() - start, 2)

# Évaluer
preds = rf_baseline.predict(X_test)
baseline_f1 = round(f1_score(y_test, preds), 4)
baseline_acc = round(accuracy_score(y_test, preds), 4)

print(f"\nDataset      : Heart Disease (303 lignes)")
print(f"Temps        : {baseline_time}s")
print(f"Accuracy     : {baseline_acc}")
print(f"F1-Score     : {baseline_f1}")
print(f"\n{classification_report(y_test, preds)}")
print(f"\nCe score est notre RÉFÉRENCE à battre avec AutoML !")

MODÈLE BASELINE — Random Forest par défaut

Dataset      : Heart Disease (303 lignes)
Temps        : 0.12s
Accuracy     : 0.8852
F1-Score     : 0.8814

              precision    recall  f1-score   support

           0       0.93      0.85      0.89        33
           1       0.84      0.93      0.88        28

    accuracy                           0.89        61
   macro avg       0.89      0.89      0.89        61
weighted avg       0.89      0.89      0.89        61


Ce score est notre RÉFÉRENCE à battre avec AutoML !


In [6]:
import json
import os

os.makedirs('../reports', exist_ok=True)

baseline_results = {
    "dataset": "Heart Disease",
    "methode": "Random Forest (baseline)",
    "n_lignes": 303,
    "n_features": 13,
    "test_size": 0.2,
    "accuracy": baseline_acc,
    "f1_score": baseline_f1,
    "temps_entrainement_s": baseline_time,
    "note": "Référence à battre avec AutoML"
}

with open('../reports/baseline_scores.json', 'w') as f:
    json.dump(baseline_results, f, indent=4)

print("Scores baseline sauvegardés dans reports/baseline_scores.json")
print(json.dumps(baseline_results, indent=4))

Scores baseline sauvegardés dans reports/baseline_scores.json
{
    "dataset": "Heart Disease",
    "methode": "Random Forest (baseline)",
    "n_lignes": 303,
    "n_features": 13,
    "test_size": 0.2,
    "accuracy": 0.8852,
    "f1_score": 0.8814,
    "temps_entrainement_s": 0.12,
    "note": "R\u00e9f\u00e9rence \u00e0 battre avec AutoML"
}


In [7]:
with open('../reports/baseline_scores.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_results, f, indent=4, ensure_ascii=False)

print("Fichier corrigé !")

Fichier corrigé !
